In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.DataFrame(pd.read_csv('Mumbai_House_Dataset.csv'))
# df.shape
# df.info()
df.columns
# df.describe()
# print(df.nunique())
# df.head()
# df.isnull().sum()
# df.head()
# df.head()
# df['location'].nunique


###---------- perform PCA 

cols = ['size_sqft','bedrooms','bathrooms','floor','total_floors']
df[cols] = df[cols].astype(float)


###------------step1.  mean-distribution
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
df.iloc[:,2:7] =  scalar.fit_transform(df.iloc[:,2:7])

###-------------step2. cov_mat
cov_mat =  df.iloc[:,2:7].cov()

###-------------step3 - eigen val and eigen vector
eigen_val,eigen_vec = np.linalg.eig(cov_mat)

# eigen_vec[0]  #(1,5) -> (5,1)
# df.iloc[:,2:7]  #(n,5)
idx = np.argmax(eigen_val)
pc = eigen_vec[idx]

###-------------- step4  - transform df
transformed_df = np.dot(df.iloc[:,2:7],pc.T)

###------pca column in data
df['pca1'] = transformed_df
df.drop(cols,axis=1,inplace=True)
df.drop('property_id',axis =1,inplace=True)
df

,location,age_of_property,furnishing,parking,near_metro,amenities_score,price,pca1
0,Andheri West,8,Semi-Furnished,Yes,Yes,7,18500000,0.777409
1,Bandra East,5,Furnished,Yes,Yes,9,32000000,-0.686277
2,Goregaon West,12,Unfurnished,No,No,5,9500000,1.643903
3,Thane West,6,Semi-Furnished,Yes,Yes,8,17500000,0.032962
4,Powai,4,Furnished,Yes,Yes,9,30000000,-1.194616
5,Malad East,9,Semi-Furnished,Yes,No,7,16000000,0.764149
6,Navi Mumbai,7,Unfurnished,Yes,Yes,6,14000000,0.228183
7,Lower Parel,3,Furnished,Yes,Yes,10,55000000,-2.436103
8,Kandivali West,15,Unfurnished,No,No,5,12000000,0.811924
9,Chembur,10,Semi-Furnished,Yes,Yes,8,21000000,-0.023877


In [55]:
# 5. What visualizations would you use to understand how location impacts price?  

location_wise_price = df.groupby('location')['price'].median().sort_values(ascending=False)
# location_wise_price.plot(kind='bar',y=df['price'])
df.head()

,location,age_of_property,furnishing,parking,near_metro,amenities_score,price,pca1
0,Andheri West,8,Semi-Furnished,Yes,Yes,7,18500000,0.777409
1,Bandra East,5,Furnished,Yes,Yes,9,32000000,-0.686277
2,Goregaon West,12,Unfurnished,No,No,5,9500000,1.643903
3,Thane West,6,Semi-Furnished,Yes,Yes,8,17500000,0.032962
4,Powai,4,Furnished,Yes,Yes,9,30000000,-1.194616


In [ ]:

num_cols =  ['age_of_property','amenities_score']
cat_cols =  ['location']
bin_cols =  ['parking','near_metro']
ordi_cols = ['furnishing']


####-----------pipline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler,OrdinalEncoder

num_col_pipe = Pipeline([
    ('scaler',StandardScaler())    
])
# cat_col_pipe = Pipeline([
#     ('encoder',OneHotEncoder(drop='first',handle_unknown='ignore'))
# ])
bin_col_pipe = Pipeline([
    ('encoder',OneHotEncoder(drop='if_binary'))
])

ordi_col_pipe = Pipeline([
    ('encoder',OrdinalEncoder(categories=[['Unfurnished','Semi-Furnished','Furnished']]))
])




####------combine pipline

from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
              ('num',num_col_pipe,num_cols),
            #   ('cat',cat_col_pipe,cat_cols),
              ('bin',bin_col_pipe,bin_cols),
              ('ordi',ordi_col_pipe,ordi_cols)
])

###------ feature selection
from sklearn.model_selection import train_test_split
X = df.drop(columns=['price','location'])
Y = df['price']

# ###---------x-y split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)


# X_train
# ###----------------- fit transform -----pipline
X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)
X_train
###----------multi-linear regresion
from sklearn.linear_model import LinearRegression

model = LinearRegression()

# ###-------model training
model.fit(X_train,Y_train)
y_p = model.predict(X_test)

# ###----------model evaluation
from sklearn.metrics import r2_score
r2_score(y_p,Y_test)




0.5590246429848484